### Creating Initial DataFrame

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window

spark = SparkSession.builder.getOrCreate()

# =========================================
# 🔹 Initial Dataset (60 records)
# =========================================

data = [(1, 'Eva', 2450), (2, 'Charlie', 3097), (3, 'Ivy', 1953), (4, 'Bob', 3924), (5, 'Bob', 4746), (6, 'Frank', 3120), (7, 'Frank', 1431), (8, 'Frank', 801), (9, 'Jack', 4388), (10, 'Eva', 3268), (11, 'Ivy', 2794), (12, 'Jack', 4153), (13, 'Alice', 4410), (14, 'Ivy', 2865), (15, 'Frank', 2337), (16, 'Grace', 2575), (17, 'Charlie', 2446), (18, 'Bob', 2462), (19, 'Frank', 682), (20, 'Charlie', 819), (21, 'Eva', 4654), (22, 'Jack', 3449), (23, 'Ivy', 4630), (24, 'Eva', 3654), (25, 'Bob', 2662), (26, 'Frank', 741), (27, 'Ivy', 3688), (28, 'Jack', 4729), (29, 'Alice', 2930), (30, 'Charlie', 1085), (31, 'David', 4096), (32, 'Helen', 523), (33, 'Jack', 868), (34, 'Eva', 3094), (35, 'Charlie', 4403), (36, 'Bob', 1953), (37, 'Frank', 1996), (38, 'Alice', 1930), (39, 'Grace', 4408), (40, 'Alice', 4145), (41, 'David', 2478), (42, 'Charlie', 3403), (43, 'David', 2553), (44, 'Grace', 2456), (45, 'Charlie', 3523), (46, 'Frank', 2434), (47, 'Charlie', 2844), (48, 'Jack', 3830), (49, 'Frank', 1889), (50, 'Grace', 3858), (51, 'Eva', 4345), (52, 'Alice', 1584), (53, 'Charlie', 1359), (54, 'Bob', 3538), (55, 'Charlie', 983), (56, 'Frank', 696), (57, 'Alice', 675), (58, 'Alice', 3153), (59, 'Alice', 1802), (60, 'Eva', 4376)]

df = spark.createDataFrame(data, ["id","name","amount"])
df.display()

id,name,amount
1,Eva,2450
2,Charlie,3097
3,Ivy,1953
4,Bob,3924
5,Bob,4746
6,Frank,3120
7,Frank,1431
8,Frank,801
9,Jack,4388
10,Eva,3268


### Create Delta Table

In [0]:
path='/Volumes/workspace/default/deltatable'
df.write.format("delta") \
    .mode("overwrite") \
    .save(path)

### Read and verify Delta Table

In [0]:
delta_df = spark.read.format('delta').load(path)
delta_df.show()
delta_df.count()

+---+-------+------+
| id|   name|amount|
+---+-------+------+
|  1|    Eva|  2450|
|  2|Charlie|  3097|
|  3|    Ivy|  1953|
|  4|    Bob|  3924|
|  5|    Bob|  4746|
|  6|  Frank|  3120|
|  7|  Frank|  1431|
|  8|  Frank|   801|
|  9|   Jack|  4388|
| 10|    Eva|  3268|
| 11|    Ivy|  2794|
| 12|   Jack|  4153|
| 13|  Alice|  4410|
| 14|    Ivy|  2865|
| 15|  Frank|  2337|
| 16|  Grace|  2575|
| 17|Charlie|  2446|
| 18|    Bob|  2462|
| 19|  Frank|   682|
| 20|Charlie|   819|
+---+-------+------+
only showing top 20 rows


60

### Insert New Records

In [0]:
new_data = [
    (61, "Alice", 5000),
    (62, "Bob", 1200),
    (63, "Zara", 7000)
]
delta_df=spark.createDataFrame(new_data, ["id","name","amount"])
delta_df.write.format("delta") \
    .mode("append") \
    .save(path)

In [0]:
spark.read.format('delta').load(path).count()

63

### Update Existing Record

In [0]:
from delta.tables import DeltaTable
delta_df=DeltaTable.forPath(spark,path)
delta_df.update(
    condition="name='bob'",
    set={'amount':'amount+500'}
)

DataFrame[num_affected_rows: bigint]

In [0]:
spark.read.format('delta').load(path).filter('id=2').display()

id,name,amount
2,Charlie,3097


### MERGE INTO

### CREATE INCREMENTAL DATA

In [0]:
merge_data = [
    (10, "Eva", 5000),   # existing → update
    (20, "Charlie", 1200), # existing → update
    (63, "David", 2200)  # new → insert
]

merge_df= spark.createDataFrame(merge_data, ["id", "name", "amount"])

In [0]:
from delta.tables import DeltaTable

target_table = DeltaTable.forPath(spark, path)
source_df = merge_df

target_table.alias("target").merge(
    source_df.alias("source"),
    "target.id = source.id"
).whenMatchedUpdate(
    set={'name': 'source.name', 'amount': 'source.amount'}
).whenNotMatchedInsert(
    values={'id': 'source.id', 'name': 'source.name', 'amount': 'source.amount'}
).execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
whole_df=spark.read.format('delta').load(path).display()
spark.read.format('delta').load(path).count()

id,name,amount
1,Eva,2450
2,Charlie,3097
3,Ivy,1953
4,Bob,3924
5,Bob,4746
6,Frank,3120
7,Frank,1431
8,Frank,801
9,Jack,4388
11,Ivy,2794


63

### Delete Invalid Records

In [0]:
delta_df.delete('amount<1500')

DataFrame[num_affected_rows: bigint]

In [0]:
spark.read.format('delta').load(path).count()

50

### SCHEMA EVOLUTION

In [0]:
new_schema_data = [
    (64, "Helen", 1800, "Retail"),
    (10, "Eva", 7000, "Premium")
]
whole_df= spark.createDataFrame(new_schema_data, ["id", "name", "amount", "category"])
whole_df.display()

id,name,amount,category
64,Helen,1800,Retail
10,Eva,7000,Premium


In [0]:
delta_df = spark.read.format("delta").load(path)
delta_df.printSchema()
delta_df.display()

root
 |-- id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- amount: long (nullable = true)



id,name,amount
1,Eva,2450
2,Charlie,3097
3,Ivy,1953
4,Bob,3924
5,Bob,4746
6,Frank,3120
9,Jack,4388
11,Ivy,2794
12,Jack,4153
13,Alice,4410


df = spark.read.format("delta").load(path)
df.printSchema()

### HISTORY

In [0]:
spark.sql(f"DESCRIBE HISTORY delta.`{path}`").display(truncate=False)

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
66,2026-04-17T10:15:03.000Z,77333947973879,22pa1a4506@vishnu.edu.in,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(1813139242436798),b98ab8a2-7cc8-4194-9c62-4522fe74e433,0417-085316-zrzqoky8-v2n,65,SnapshotIsolation,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 1692, p25FileSize -> 1573, numDeletionVectorsRemoved -> 1, minFileSize -> 1573, numAddedFiles -> 1, maxFileSize -> 1573, p75FileSize -> 1573, p50FileSize -> 1573, numAddedBytes -> 1573)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
65,2026-04-17T10:15:01.000Z,77333947973879,22pa1a4506@vishnu.edu.in,DELETE,"Map(predicate -> [""(amount#28299L < 1500)""])",null,List(1813139242436798),b98ab8a2-7cc8-4194-9c62-4522fe74e433,0417-085316-zrzqoky8-v2n,64,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 1138, numDeletionVectorsUpdated -> 0, numDeletedRows -> 13, scanTimeMs -> 868, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 270)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
64,2026-04-17T10:14:59.000Z,77333947973879,22pa1a4506@vishnu.edu.in,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(1813139242436798),c4304f36-f226-40d7-bba5-8b1d35005bcd,0417-085316-zrzqoky8-v2n,63,SnapshotIsolation,false,"Map(numRemovedFiles -> 5, numRemovedBytes -> 5816, p25FileSize -> 1692, numDeletionVectorsRemoved -> 2, minFileSize -> 1692, numAddedFiles -> 1, maxFileSize -> 1692, p75FileSize -> 1692, p50FileSize -> 1692, numAddedBytes -> 1692)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
63,2026-04-17T10:14:57.000Z,77333947973879,22pa1a4506@vishnu.edu.in,MERGE,"Map(predicate -> [""(id#27895L = id#27898L)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(1813139242436798),c4304f36-f226-40d7-bba5-8b1d35005bcd,0417-085316-zrzqoky8-v2n,62,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 3, numTargetBytesAdded -> 3072, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 2, numTargetRowsMatchedUpdated -> 3, executionTimeMs -> 2317, materializeSourceTimeMs -> 85, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 925, numTargetRowsUpdated -> 3, numOutputRows -> 3, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 3, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1217)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
62,2026-04-17T10:14:52.000Z,77333947973879,22pa1a4506@vishnu.edu.in,UPDATE,"Map(predicate -> [""(name#27768 = bob)""])",null,List(1813139242436798),a9320ff6-c827-4e6b-ba74-84f9b82cff53,0417-085316-zrzqoky8-v2n,61,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 215, numDeletionVectorsUpdated -> 0, scanTimeMs -> 215, numAddedFiles -> 0, numUpdatedRows -> 0, numAddedBytes -> 0, rewriteTimeMs -> 0)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
61,2026-04-17T10:14:51.000Z,77333947973879,22pa1a4506@vishnu.edu.in,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [])",null,List(1813139242436798),5a248db5-6180-46b0-96ba-be9fff26fb9f,0417-085316-zrzqoky8-v2n,60,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 3, numOutputBytes

### Query Older Version

In [0]:
old_df = spark.read.format("delta") \
    .option("versionAsOf", 0) \
    .load(path)
old_df1 = spark.read.format("delta") \
    .option("versionAsOf", 1) \
    .load(path)

old_df.display()
old_df1.display()

id,name,amount
1,Eva,2450
2,Charlie,3097
3,Ivy,1953
4,Bob,3924
5,Bob,4746
6,Frank,3120
7,Frank,1431
8,Frank,801
9,Jack,4388
10,Eva,3268


id,name,amount
1,Eva,2450
2,Charlie,3097
3,Ivy,1953
4,Bob,3924
5,Bob,4746
6,Frank,3120
7,Frank,1431
8,Frank,801
9,Jack,4388
10,Eva,3268


### Restore Table

In [0]:
spark.sql(f"""
RESTORE TABLE delta.`{path}`
TO VERSION AS OF 0
""")

DataFrame[table_size_after_restore: bigint, num_of_files_after_restore: bigint, num_removed_files: bigint, num_restored_files: bigint, removed_files_size: bigint, restored_files_size: bigint]

### Verify Rollback

In [0]:
spark.read.format("delta").load(path).count()

60

### VALIDATION

### Total Count

In [0]:
df_final = spark.read.format("delta").load(path)
df_final.count()

60

### Check Duplicates

In [0]:
df_final.groupBy("id").count().filter("count > 1").show()

+---+-----+
| id|count|
+---+-----+
+---+-----+



### Verify Updates 

In [0]:
df_final.filter("id IN (10,20)").show()

+---+-------+------+
| id|   name|amount|
+---+-------+------+
| 10|    Eva|  3268|
| 20|Charlie|   819|
+---+-------+------+



### Data Integrity Check

In [0]:
df_final.describe().show()

+-------+-----------------+-----+------------------+
|summary|               id| name|            amount|
+-------+-----------------+-----+------------------+
|  count|               60|   60|                60|
|   mean|             30.5| NULL|2768.9666666666667|
| stddev|17.46424919657298| NULL|1262.6158609923204|
|    min|                1|Alice|               523|
|    max|               60| Jack|              4746|
+-------+-----------------+-----+------------------+

